In [ ]:
# most recent particle file analyzer

findex = 230 #144 #121 #120 #22 
step_choice = "last" #"first"

In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun

%matplotlib widget

%load_ext autoreload
%autoreload 2 

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps}

my_steps = tstep_choices[step_choice]
boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"


In [ ]:
# Config Data

d = run.config_data(my_steps, "both")

In [ ]:
print(run.stride)

In [ ]:
species_idx = 2
species = ["BAD SPECIES", "electron", "ion"]

df = d.particles[species_idx].to_dataframe()
species = species[species_idx]

ann_size(df)
add_gamma(df)

In [ ]:
roi = fq_cut(d.fields, 0, [0, run.xlim, 0, run.ylim]) 

In [ ]:
from matplotlib.path import Path

def choose_rect(df, rect):
    [x1, x2, y1, y2] = rect
    return df.loc[(x1 < df['x']) & (df['x'] < x2) & (y1 < df['y']) & (df['y'] < y2)]

def choose_circle(df, x0, y0, r):
    return df.loc[(df['x'] - x0) ** 2 + (df['y'] - y0) ** 2 < r ** 2]

def choose_circle_slice(df, x0, y0, r1, r2):
    df['dist'] = np.sqrt((df['x'] - x0) ** 2 + (df['y'] - y0) ** 2)

    return df.loc[(r1 < df['dist']) & (df['dist'] < r2)]

def choose_ellipse(df, x0, y0, a, b):
    return df.loc[((df['x'] - x0) / a) ** 2 + ((df['y'] - y0) / b) ** 2 < 1]

def choose_quad(df, points):

    # Create a Path object from the points
    path = Path(points)

    # Create an array of (x, y) points from the dataframe
    points_array = df[['x', 'y']].values

    # Use the Path object to determine which points are inside the quadrilateral
    mask = path.contains_points(points_array)

    # Return the dataframe rows that are inside the quadrilateral
    return df[mask]

In [ ]:
print(run.xlim / 2)

In [ ]:

chosen_sets = [
    choose_rect(df, [0, 1000, 2500, 5500])
]

#chosen_sets = [
#    choose_quad(df, [(337, 1255), (420, 1255), (417, 1380), (334, 1380)])
#]

chosen_sets = [
    choose_quad(df, [(320, 1255), (430, 1255), (417, 1380), (334, 1380)])
]

chosen_sets = [
    choose_quad(df, [(400, 887), (431, 887), (478, 510), (367, 510)])
]

chosen_sets = [
    choose_quad(df, [(402, 975), (420, 975), (473, 576), (376, 576)])
]

#chosen_sets = [
#    choose_quad(df, [(385, 710), (452, 710), (473, 576), (376, 576)])
#]

colors = ['blue', 'red', 'green', 'pink', 'yellow', 'purple', 'orange', 'cyan', 'magenta', 'black']


In [ ]:
print(len(chosen_sets[0]))

In [ ]:


xdiff = 50
crect = [run.xlim / 2 - xdiff, run.xlim / 2 + xdiff, 0, run.ylim / 2]
crect = [xdiff, run.xlim - xdiff, 0, run.ylim / 2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (10, 5))
fig.suptitle(run.boilerplate + f"\n t = {my_steps[0]}")

# Left Plot

for i in range(len(chosen_sets)):
    chosen = chosen_sets[i]
    alphas = [1, 1, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05]
    ax1.scatter(chosen['x'], chosen['y'], s = 0.1, c = colors[i] , alpha = alphas[i])


plot_streamline(ax1, roi, roi.bx, roi.by, 0.75, 'w')
mesh = plot_fq3(roi, roi.dens, plot = ax1, vabsmax = 4)
#ax1.contour(cs.x, cs.y, chi, [0], colors = 'green')

bthresh = 0.9
ax1.contour(roi.x, roi.y, roi.pb, [bthresh], colors = 'green')

ax1.set_xlim(crect[0], crect[1])
ax1.set_ylim(crect[2], crect[3])
ax1.set_aspect('equal')

cbar = fig.colorbar(mesh, ax = ax1)
cbar.set_label("$n / n_0$")

# Right Plot

log_bins = 10 ** np.linspace(-4, 4, 100) #300

for i in range(len(chosen_sets)):
    chosen = chosen_sets[i]
    add_gamma(chosen)
    chosen['beta_w'] = chosen['w'] / chosen['gamma']

    hist, bins = np.histogram((chosen['gamma'] - 1) / run.sigma, bins = log_bins, density = True)
    
    mean = np.mean(chosen['beta_w'])

    ax2.loglog(bins[1:], hist, color = colors[i], label = fr'{len(chosen):.2e} prtls ($\beta_w =$ {mean:.2f})')


ax2.set_xlabel("$(\gamma - 1) / \sigma$")
ax2.set_ylabel("$dN / d\gamma$")
ax2.set_title(f"{species} Spectra")
ax2.grid(visible = True)
ax2.axvline(1, color = 'black', linestyle = '--', label = '$\gamma - 1 = \sigma$')
ax2.axvline(1 / run.sigma, color = 'orange', linestyle = '--', label = '$\gamma - 1 = 1$')
#ax2.legend()
plt.tight_layout()

In [ ]:
# Eckart Frame

total_rmass = np.sum(chosen['gamma'])

bulk_u = np.sum(chosen['u']) / total_rmass
bulk_v = np.sum(chosen['v']) / total_rmass
bulk_w = np.sum(chosen['w']) / total_rmass 

beta_A = run.beta_A

print(f"bulk u / v_A = {bulk_u / beta_A}, bulk v / v_A = {bulk_v / run.beta_A}, bulk w / v_A = {bulk_w / run.beta_A}")

In [ ]:
def g2b(gamma):
    return np.sqrt(1 - 1 / gamma ** 2)

def b2g(beta):
    return 1 / np.sqrt(1 - beta ** 2)

def fourvel2beta(u):
    return u / np.sqrt(1 + u ** 2)


In [ ]:
# COM Frame (warning overwrites the above)

total_rmass = np.sum(chosen['gamma'])

bulk_u = np.sum(chosen['u']) / total_rmass
bulk_v = np.sum(chosen['v']) / total_rmass
bulk_w = np.sum(chosen['w']) / total_rmass 

v_A = run.beta_A / (1 - run.beta_A ** 2) ** 0.5

print(f"bulk u / v_A = {bulk_u / v_A}, bulk v / v_A = {bulk_v / v_A}, bulk w / v_A = {bulk_w / v_A}")

In [ ]:
from scipy.stats import norm

fig, ax = plt.subplots(figsize=(10, 5))

# Plot histogram as lines
hist_u, bins_u = np.histogram(chosen['u'], bins=50, density=True)
hist_w, bins_w = np.histogram(chosen['w'], bins=50, density=True)
bin_centers_u = (bins_u[:-1] + bins_u[1:]) / 2
bin_centers_w = (bins_w[:-1] + bins_w[1:]) / 2

ax.plot(bin_centers_u, hist_u, label='u', color='blue')
ax.plot(bin_centers_w, hist_w, label='w', color='red')

# Fit Gaussian for 'u'
mu_u, std_u = norm.fit(chosen['u'])
p_u = norm.pdf(bin_centers_u, mu_u, std_u)
ax.plot(bin_centers_u, p_u, 'b--', label=f'Fit u: $\mu={mu_u:.2f}$, $\sigma={std_u:.2f}$')

# Fit Gaussian for 'w'
mu_w, std_w = norm.fit(chosen['w'])
p_w = norm.pdf(bin_centers_w, mu_w, std_w)
ax.plot(bin_centers_w, p_w, 'r--', label=f'Fit w: $\mu={mu_w:.2f}$, $\sigma={std_w:.2f}$')

ax.set_xlabel('Value')
ax.set_ylabel('Frequency')
ax.set_title('Histogram of u and w with Gaussian Fits')
ax.legend()
plt.show()

In [ ]:
def y_lorentz_boost(beta_y_frame, df):
    df['beta_x'] = df['u'] / df['gamma']
    df['beta_y'] = df['v'] / df['gamma']
    df['beta_z'] = df['w'] / df['gamma']

    bulk_gamma = b2g(beta_y_frame)

    df['denom'] = (1 - beta_y_frame * df['beta_y'])

    df['beta_xp'] = (df['beta_x'] / bulk_gamma) / df['denom']
    df['beta_yp'] = (df['beta_y'] / bulk_gamma - beta_y_frame) / df['denom']
    df['beta_zp'] = (df['beta_z'] / bulk_gamma) / df['denom']

    df['gammap'] = 1 / np.sqrt(1 - df['beta_xp'] ** 2 - df['beta_yp'] ** 2 - df['beta_zp'] ** 2)

    df['up'] = df['beta_xp'] * df['gammap']
    df['vp'] = df['beta_yp'] * df['gammap']
    df['wp'] = df['beta_zp'] * df['gammap']
    
    return df

In [ ]:
# Frame Transformation

#chosen = y_lorentz_boost(-run.beta_A, chosen)
chosen = y_lorentz_boost(fourvel2beta(bulk_v), chosen)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bins = 50

# Calculate histogram and Gaussian fit for 'up'
hist_up, bins_up = np.histogram(chosen['up'], bins=bins, density=True)
bin_centers_up = (bins_up[:-1] + bins_up[1:]) / 2
mu_up, std_up = norm.fit(chosen['up'])
p_up = norm.pdf(bin_centers_up, mu_up, std_up)

# Plot histogram and Gaussian fit for 'up'
ax.plot(bin_centers_up, hist_up, label='$u$\'', color='blue')
ax.plot(bin_centers_up, p_up, 'b--', label=f'Fit up: $\mu={mu_up:.2f}$, $\sigma={std_up:.2f}$')

# Calculate histogram and Gaussian fit for 'vp'
hist_vp, bins_vp = np.histogram(chosen['vp'], bins=bins, density=True)
bin_centers_vp = (bins_vp[:-1] + bins_vp[1:]) / 2
mu_vp, std_vp = norm.fit(chosen['vp'])
p_vp = norm.pdf(bin_centers_vp, mu_vp, std_vp)

# Plot histogram and Gaussian fit for 'vp'
ax.plot(bin_centers_vp, hist_vp, label='$v$\'', color='green')
ax.plot(bin_centers_vp, p_vp, 'g--', label=f'Fit vp: $\mu={mu_vp:.2f}$, $\sigma={std_vp:.2f}$')

# Calculate histogram and Gaussian fit for 'wp'
hist_wp, bins_wp = np.histogram(chosen['wp'], bins=bins, density=True)
bin_centers_wp = (bins_wp[:-1] + bins_wp[1:]) / 2
mu_wp, std_wp = norm.fit(chosen['wp'])
p_wp = norm.pdf(bin_centers_wp, mu_wp, std_wp)

# Plot histogram and Gaussian fit for 'wp'
ax.plot(bin_centers_wp, hist_wp, label='$w$\'', color='red')
ax.plot(bin_centers_wp, p_wp, 'r--', label=f'Fit wp: $\mu={mu_wp:.2f}$, $\sigma={std_wp:.2f}$')

ax.set_xlabel('Value')
ax.set_ylabel('Frequency')
ax.set_title('Histograms of vp, up, and wp with Gaussian Fits')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
print(2 + 2)

In [ ]:
from scipy.optimize import curve_fit

# Define your arbitrary function

import matplotlib as mpl

mpl.rcParams['font.size'] = 16

def maxwell_juttner(gamma, coeff, theta):
    #TODO
    return coeff * gamma ** 2 * g2b(gamma) * np.exp(-gamma / theta)

fig, ax = plt.subplots(figsize=(10, 5))

# Plot histogram as lines
#hist_gammap, bins_gammap = np.histogram(chosen['gammap'], bins=50, density=True)
#bin_centers_gammap = (bins_gammap[:-1] + bins_gammap[1:]) / 2

def get_log_histogram(data, num_bins=50):
    # 1. Create bins evenly spaced in log-space
    # We use the min and max of the actual data
    bins = np.logspace(np.log10(data.min()), np.log10(data.max()), num_bins + 1)
    
    # 2. Compute the histogram
    # 'density=True' ensures the area under the curve is 1
    hist, bin_edges = np.histogram(data, bins=bins, density=True)
    
    # 3. Calculate bin centers (geometric mean is often better for log-bins)
    # But arithmetic mean of edges is standard for curve_fit
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    return hist, bin_centers, bin_edges

# Usage:
hist_gammap, bin_centers_gammap, bins_gammap = get_log_histogram(chosen['gammap'], num_bins=30)

ax.loglog(bin_centers_gammap - 1, hist_gammap, label='simulation spectrum', color='purple') #[8e23, 0.02]

species_name = {1: "Electron", 2: "Ion"}[species_idx]

np.save(f"gammap_bin_centers_{species_idx}.npy", bin_centers_gammap)
np.save(f"gammap_hist_{species_idx}.npy", hist_gammap)

# Fit arbitrary function to 'gammap'
popt, pcov = curve_fit(maxwell_juttner, bin_centers_gammap, hist_gammap, p0 = [1e176, 0.0025], bounds=(0, [np.inf, np.inf]))
p_gammap = maxwell_juttner(bin_centers_gammap, *popt)
#ax.loglog(bin_centers_gammap - 1, p_gammap, '--', color='purple', label=fr'$T / mc^2$ = {popt[1]:.4f} Maxwell-Jüttner')



plt.title(f"{species_name} Spectrum in Exhaust Flow Frame")
ax.set_xlabel('$\gamma\' - 1$')
ax.set_ylabel('$f(\gamma\')$')
ax.legend()
ax.grid()

#save_polished_fig("mjf")

plt.show()

In [ ]:
for i, chosen in enumerate(chosen_sets):
    print(f"Set {i}: mean(gamma - 1) = {np.mean(chosen['gamma'] - 1):.6f}")

In [ ]:
print(0 / 0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import kn  # Modified Bessel function of second kind

# 1. Robust MJ Function
def maxwell_juttner(gamma, A, theta):
    """Standard MJ distribution."""
    # beta = sqrt(1 - 1/gamma^2)
    beta = np.sqrt(1 - 1/gamma**2)
    # The normalization involves K2(1/theta)
    den = theta * kn(2, 1/theta)
    return A * (gamma**2 * beta / den) * np.exp(-gamma / theta)

# 2. Log-Log Wrapper for Curve Fit
def log_mj_fit(log_gamma_minus_1, log_A, log_theta):
    """
    Fits log10(f(gamma)) as a function of log10(gamma - 1).
    We use log parameters to keep the optimizer in a well-behaved space.
    """
    # Convert log inputs back to linear for the MJ formula
    gamma = 10**log_gamma_minus_1 + 1
    A = 10**log_A
    theta = 10**log_theta
    
    # Calculate MJ and return the log
    val = maxwell_juttner(gamma, A, theta)
    return np.log10(val + 1e-30) # Epsilon to avoid log(0)

# --- Data Preparation ---
# Using your existing get_log_histogram logic
hist_gammap, bin_centers_gammap, bins_gammap = get_log_histogram(chosen['gammap'], num_bins=100)

# Filter for valid data (must be gamma > 1 and density > 0 for log-log)
mask = (bin_centers_gammap > 1) & (hist_gammap > 0)
x_data_log = np.log10(bin_centers_gammap[mask] - 1)
y_data_log = np.log10(hist_gammap[mask])

# --- Perform the Fit ---
# Initial guesses in log10 space
# If theta ~ 0.0025, log_theta ~ -2.6
initial_guess_log = [0.0, -2.6] 

try:
    popt_log, pcov_log = curve_fit(log_mj_fit, x_data_log, y_data_log, p0=initial_guess_log)
    
    # Convert optimized log parameters back to linear
    fit_A = 10**popt_log[0]
    fit_theta = 10**popt_log[1]
    print(f"Fit Result: A={fit_A:.2e}, theta={fit_theta:.6f}")
    
    # Generate the fit curve for plotting
    p_gammap = maxwell_juttner(bin_centers_gammap, fit_A, fit_theta)
    
except Exception as e:
    print(f"Fit failed: {e}")
    p_gammap = None

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

# Plot actual data
ax.loglog(bin_centers_gammap - 1, hist_gammap, 'o', ms=4, alpha=0.6, label='Simulation Spectrum', color='purple')

# Plot fit
if p_gammap is not None:
    ax.loglog(bin_centers_gammap - 1, p_gammap, '--', lw=2, color='darkorange', 
              label=fr'MJ Fit: $\theta$ = {fit_theta:.5f}')
    
test_A = 1.1
test_theta = 0.004
test_gammap = maxwell_juttner(bin_centers_gammap, test_A, test_theta)

ax.loglog(bin_centers_gammap - 1, test_gammap, '--', lw=2, color='cyan', label=fr'Test MJ: $\theta$ = {test_theta:.4f}')

ax.set_xlabel(r'$\gamma - 1$')
ax.set_ylabel(r'$f(\gamma)$')
ax.set_title('Log-Log Maxwell-Jüttner Fit')
ax.legend()
ax.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

In [ ]:
print(run.xlim)

In [ ]:
# Calculate the pitch angle and energy for each particle
chosen['v_mag'] = np.sqrt(chosen['u']**2 + chosen['v']**2 + chosen['w']**2)
chosen['B_mag'] = np.sqrt(chosen['bx']**2 + chosen['by']**2 + chosen['bz']**2)
chosen['v_dot_B'] = chosen['u']*chosen['bx'] + chosen['v']*chosen['by'] + chosen['w']*chosen['bz']
chosen['pitch_angle'] = np.degrees(np.arccos(chosen['v_dot_B'] / (chosen['v_mag'] * chosen['B_mag'])))
chosen['energy'] = chosen['gamma'] - 1

# Convert pitch angle to radians for polar plot
chosen['pitch_angle_rad'] = np.radians(chosen['pitch_angle'])

# Compute 2D histogram for pitch angle and energy with improved resolution
hist, xedges, yedges = np.histogram2d(chosen['pitch_angle'], chosen['energy'], bins=[100, 100], density=True)

# Convert pitch angle to radians for polar plot
chosen['pitch_angle_rad'] = np.radians(chosen['pitch_angle'])

# Plot the 2D histogram as a polar density plot
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 5))
theta, r = np.meshgrid(np.radians(xedges), yedges)
pcm = ax.pcolormesh(theta, r, hist.T, cmap='viridis')
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label('Density')

ax.set_xlabel('Pitch Angle')
ax.set_ylabel('$\gamma - 1$')
ax.set_title('Density of Pitch Angle vs Energy')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate sin^2 of the pitch angle
chosen['sin2_pitch_angle'] = np.sin(chosen['pitch_angle_rad']) ** 2

# Compute 2D histogram for sin^2(pitch angle) and gamma
hist, xedges, yedges = np.histogram2d(chosen['sin2_pitch_angle'], chosen['gamma'], bins=[100, 100], density=True)

# Plot the 2D histogram as a density plot
fig, ax = plt.subplots(figsize=(10, 5))
pcm = ax.pcolormesh(xedges, yedges, hist.T, cmap='viridis')
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label('Density')

ax.set_xlabel('sin2(Pitch Angle)')
ax.set_ylabel('Gamma')
ax.set_title('Density of sin2(Pitch Angle) vs Gamma')

plt.tight_layout()
plt.show()